# 17 感知机 Perceptron

感知机是早期神经网络模型，也是理解线性分类和神经元的好入口。它只能保证在线性可分数据上收敛。


## 0. 学习目标和阅读地图

感知机是神经网络的历史起点。你需要掌握：

1. 线性分类器如何用符号函数做分类。
2. 只有分错样本才触发更新。
3. 线性可分时为什么会收敛。
4. 感知机和逻辑回归有什么区别。


## 1. 数学逻辑

感知机预测：

$$\hat y = sign(w^Tx+b)$$

当样本被分错时更新：

$$w \leftarrow w + \eta y_i x_i$$

$$b \leftarrow b + \eta y_i$$

其中标签通常取 `-1` 和 `1`。


## 1.1 推导拆开看：分错才更新

如果标签 `y in {-1, 1}`，预测符号是：

$$sign(w^Tx+b)$$

分类正确意味着：

$$y(w^Tx+b)>0$$

如果分错，则：

$$y(w^Tx+b)\le 0$$

更新：

$$w \leftarrow w + \eta yx$$

会让 `y(w^Tx+b)` 增大，使这个样本下次更可能被分对。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager


def setup_chinese_font():
    candidates = [
        'PingFang SC',
        'Heiti SC',
        'Songti SC',
        'Arial Unicode MS',
        'Noto Sans CJK SC',
        'Noto Sans SC',
        'SimHei',
        'Microsoft YaHei',
        'WenQuanYi Micro Hei',
    ]
    available = {font.name for font in font_manager.fontManager.ttflist}
    for font in candidates:
        if font in available:
            existing = [name for name in plt.rcParams['font.sans-serif'] if name != font]
            plt.rcParams['font.family'] = 'sans-serif'
            plt.rcParams['font.sans-serif'] = [font] + existing
            break
    else:
        print('Warning: no Chinese font found. Install Noto Sans CJK SC or SimHei if Chinese text is missing in plots.')
    plt.rcParams['axes.unicode_minus'] = False


setup_chinese_font()

from sklearn.datasets import make_blobs
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X, y01 = make_blobs(n_samples=180, centers=2, cluster_std=1.0, random_state=42)
y = np.where(y01 == 1, 1, -1)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


## 1.2 感知机和神经元

感知机可以看作一个最简单神经元：输入特征乘权重，加偏置，再经过阶跃函数输出类别。

现代神经网络把阶跃函数换成可微激活函数，并堆叠很多层，从而获得更强表达能力。


In [ ]:
# 从零实现感知机
w = np.zeros(X_train_s.shape[1])
b = 0.0
lr = 0.1
mistakes_history = []

for epoch in range(20):
    mistakes = 0
    for xi, yi in zip(X_train_s, y_train):
        pred = 1 if xi @ w + b >= 0 else -1
        if pred != yi:
            w += lr * yi * xi
            b += lr * yi
            mistakes += 1
    mistakes_history.append(mistakes)
    print(f'epoch {epoch+1:2d} | mistakes={mistakes}')

pred = np.where(X_test_s @ w + b >= 0, 1, -1)
print('从零感知机 accuracy:', round(accuracy_score(y_test, pred), 3))
plt.plot(mistakes_history, marker='o')
plt.title('每轮误分类数量')
plt.xlabel('epoch')
plt.ylabel('mistakes')
plt.show()


## 1.3 从零实现代码怎么读

每个 epoch 扫一遍训练集。只有当 `pred != yi` 时才更新。

`mistakes_history` 很适合观察训练是否收敛。如果数据线性可分，误分类数量通常会降到 0；如果不可分，可能一直波动。


In [ ]:
model = Perceptron(max_iter=1000, eta0=0.1, random_state=42)
model.fit(X_train_s, y_train)
print('sklearn Perceptron accuracy:', round(accuracy_score(y_test, model.predict(X_test_s)), 3))


In [ ]:
# 诊断：画出感知机学到的线性边界
plt.scatter(X_train_s[:,0], X_train_s[:,1], c=y_train, cmap='coolwarm', edgecolor='k')
xs = np.linspace(X_train_s[:,0].min()-0.5, X_train_s[:,0].max()+0.5, 100)
ys = -(w[0] * xs + b) / (w[1] + 1e-12)
plt.plot(xs, ys, color='black')
plt.title('从零感知机的线性决策边界')
plt.xlabel('x1')
plt.ylabel('x2')
plt.show()


## 2.1 如何诊断感知机

如果误分类数量不能下降，可能有几种原因：

- 数据不是线性可分。
- 学习率太大导致震荡。
- 特征没有标准化。
- 样本中有异常点或错误标签。


## 2. 常见误区

- 感知机不是概率模型，不输出可靠概率。
- 数据不可线性分时，它可能一直震荡。
- 它是理解神经网络的入口，但表达能力远弱于多层网络。

## 3. 小实验

- 把 `cluster_std` 调大，让数据不可线性分。
- 改学习率 `lr`。
- 对比逻辑回归，看 loss 和概率输出差异。


## 5. 复习清单

- 感知机是线性分类器。
- 它只在分错时更新。
- 不输出概率。
- 它是理解 MLP 的入口，但表达能力有限。
